<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/00_start_here/01_ml_step_by_step_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗺️ The ML Step-by-Step Guide (2nd edition)

**A reusable map for any tabular competition or project.** Copy this notebook, rename it after your challenge
(`Zindi_<challenge>_v1.ipynb`), and fill each section. Every phase explains **why** it exists, **what** to do, and the
**classic traps**. Working examples of every step live in the tabular quests and applications.

```
 0. Frame the problem ──▶ 1. Setup ──▶ 2. Load ──▶ 3. EDA ──▶ 4. Validation strategy
                                                                   │
 9. Export & submit ◀── 8. Tune & ensemble ◀── 7. Iterate ◀── 6. Baseline ◀── 5. Features & preprocessing
```

> **What changed since the first edition?** A *problem framing* step, a validation strategy chosen **before** any
> modelling, preprocessing inside a scikit-learn `Pipeline` (no leakage, one object to export), a **dumb baseline**
> first, cross-validation instead of a single split, an experiment leaderboard, a submission checklist, and the
> installation now uses **uv**.

# 0. Frame the problem 🎯

*Why?* Most failed projects solve the wrong problem. Five minutes here save days later.

Answer these questions in this cell (double-click to edit):

| Question | Your answer |
|---|---|
| What do we predict (the **target**)? One row = one what? | … |
| Task type: binary / multi-class classification, regression, ranking…? | … |
| **Metric** used for scoring (read the competition page!) | … |
| What does the submission file look like (`SampleSubmission.csv`)? | … |
| Is there a **time** dimension (rows ordered by date)? **Groups** (same customer / patient on several rows)? | … |
| What would a human expert look at to make this prediction? | … |

💡 *The last question is your first source of feature ideas.*

# 1. Setup ⚙️

## Installation
With [uv](https://docs.astral.sh/uv/), everything is declared in `pyproject.toml` and installed in a local `.venv`:

```bash
uv sync --group tabular        # only what tabular projects need
uv add <new-library>           # add a library to the project (updates pyproject.toml + uv.lock)
```

In a notebook you can still run `!uv pip install <lib>` for a quick experiment, but prefer `uv add`
so your teammates get the same environment.

In [1]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "optuna"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [2]:
# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & modelling (scikit-learn)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge

# Gradient boosting: the kings of tabular data
# from lightgbm import LGBMClassifier, LGBMRegressor
# from xgboost import XGBClassifier, XGBRegressor
# from catboost import CatBoostClassifier, CatBoostRegressor

# Other
import joblib

pd.set_option("display.max_columns", 100)

# 2. Data loading 📥

*Why?* Know exactly what you have: files, sizes, columns, ids. Keep raw files untouched in `data/<challenge>/`.

```python
DATA = DATA_DIR / "my_challenge"          # put Train.csv, Test.csv, SampleSubmission.csv here
train = pd.read_csv(DATA / "Train.csv")
test = pd.read_csv(DATA / "Test.csv")
sample = pd.read_csv(DATA / "SampleSubmission.csv")
```

In [3]:
# Code here: load train / test / sample submission
# train.shape, test.shape, train.head()

# 3. Exploratory Data Analysis (EDA) 🔍

*Why?* To **understand** the data, **form hypotheses** and **plan** cleaning and features. Write your conclusions in
markdown cells: an EDA without conclusions is just pictures.

Checklist:
- [ ] `train.info()`, `train.describe()`, number of unique values per column
- [ ] **Target**: distribution, class imbalance (classification), skewness (regression → log?)
- [ ] **Missing values**: how many, where, is "missing" informative?
- [ ] **Duplicates** and constant columns
- [ ] **Train vs test**: do they look alike? (different distributions → careful with validation)
- [ ] Is there an `id`, a date, a group key?

## 3.1 Univariate analysis
One variable at a time: histograms for numbers, bar charts for categories.

In [4]:
# Code here

## 3.2 Bivariate & multivariate analysis
Each feature **against the target** (box plots, target rate per category), correlations between features.

In [5]:
# Code here

## 3.3 Hypotheses
- H1: …
- H2: …

# 4. Validation strategy 🛡️ (decide it **before** modelling!)

*Why?* Your local score is your compass. If it doesn't reflect the leaderboard, every decision is a coin flip.
**Your validation must mimic how the test set was built.**

| Situation | Use |
|---|---|
| Rows are independent, classification | `StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)` |
| Rows are independent, regression | `KFold(n_splits=5, shuffle=True, random_state=SEED)` |
| Several rows per customer/patient/field, test has *new* groups | `GroupKFold` (or `StratifiedGroupKFold`) |
| Test is *in the future* | `TimeSeriesSplit` or a time-based holdout |

⚠️ **Data leakage** = information from the validation/test set sneaks into training (e.g. you fit a scaler on all
data, or a feature is computed using the target). Symptom: amazing local score, poor leaderboard.

In [6]:
# Code here: define your splitter, e.g.
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 5. Feature processing & engineering 🛠️

*Why?* Models only see numbers. Good features often beat fancy models.

## 5.1 Cleaning (done on the raw DataFrames, before splitting is fine *only* if it uses no statistics)
- drop exact duplicates (`train.drop_duplicates()`), fix typos in categories, parse dates.

## 5.2 New features (write a function applied identically to train and test)
- ratios, counts, date parts (month, weekday), text length, aggregations by group…

```python
def make_features(df):
    df = df.copy()
    # df["income_per_person"] = df["income"] / df["household_size"]
    return df
```

## 5.3 Preprocessing **inside a Pipeline** (imputation, encoding, scaling)
Statistics (means, categories, scales) are learnt on the training folds only, and the whole thing exports as one object.

```python
num_cols = [...]; cat_cols = [...]
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat_cols),
])
```

💡 Tree models (LightGBM, CatBoost) don't need scaling and handle missing values natively; CatBoost even handles
categories itself.

## 5.4 Optional: class imbalance
Prefer `class_weight="balanced"` and the right metric (F1, AUC) first. Resampling (`imbalanced-learn`) must be applied
**only inside the training folds** (`imblearn.pipeline.Pipeline`).

In [7]:
# Code here

# 6. Baselines 🧱

*Why?* A **dumb baseline** (`DummyClassifier`/`DummyRegressor`) tells you the score of "no intelligence". A **simple
baseline** (logistic/ridge regression) tells you how much a real model must beat. If your fancy model can't beat them,
there's a bug.

In [8]:
lb = Leaderboard("my_challenge", metric="score", higher_is_better=True)   # change metric name & direction
# scores = cross_val_score(make_pipeline(preprocess, DummyClassifier()), X, y, cv=cv, scoring="...")
# lb.add("Dummy", scores.mean(), std=scores.std())

# 7. Iterate: models & features 🔁

Keep the structure **create → cross-validate → log** for every idea. Change **one thing at a time** so you know what
helped.

## Model #001

In [9]:
# model = make_pipeline(preprocess, LogisticRegression(max_iter=1000))
# scores = cross_val_score(model, X, y, cv=cv, scoring="...")
# lb.add("LogReg", scores.mean(), std=scores.std())

## Model #002

In [10]:
# Code here

## Models comparison

In [11]:
lb.show()

Leaderboard is empty: train a model first!


""


# 8. Hyperparameter tuning & ensembling 🎛️

*Why?* Tuning squeezes the last points out of your best models; ensembling averages away their individual errors.

- Tune the **top 2-3** models only, with `RandomizedSearchCV` or **Optuna** (smarter than a grid).
- Average the predictions (or probabilities) of different model families: often a free boost.
- Always compare with your cross-validation, never with the public leaderboard alone.

In [12]:
# Code here

# 9. Train on all data, predict, submit & export 📤

1. Refit the final pipeline on **all** training data.
2. Predict the test set and build the submission **exactly** like `SampleSubmission.csv`.
3. Save the fitted pipeline for an app (Streamlit, Gradio, FastAPI) with `joblib`.

**Submission checklist:** same columns and order as the sample · same number of rows · same ids · no missing values ·
probabilities vs labels as required.

```python
final_model.fit(X, y)
sub = sample.copy(); sub["target"] = final_model.predict(test)
make_submission(sub, "my_challenge_v1.csv", sample=sample)
joblib.dump(final_model, RUNS_DIR / "my_challenge_pipeline.joblib")
```

In [13]:
# Code here

# 10. Write it up ✍️

A short summary is what turns a notebook into a portfolio item:
- best CV score vs leaderboard score
- what worked, what didn't
- next ideas

🎓 Done? Run `checkpoint("template/first_project", xp=200)` and treat yourself!